# Step 1: Setup prerequisites

### **Set the passkey provided by your workshop instructor**

In [1]:
import os
import sys
import pymongo
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from dotenv import load_dotenv

# Add parent directory to path to import from utils
sys.path.append(os.path.join(os.path.dirname(os.getcwd())))
from utils import set_env

load_dotenv()

MONGODB_URI = os.environ.get("MONGODB_URI")

# Create a new client and connect to the server
mongodb_client = MongoClient(MONGODB_URI, appname="devrel-workshop-rag", server_api=ServerApi('1'))

# Send a ping to confirm a successful connection
try:
    mongodb_client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)

Pinged your deployment. You successfully connected to MongoDB!


# Step 2: Load the dataset

In [2]:
import json

In [3]:
with open("data/mongodb_docs.json", "r") as data_file:
    json_data = data_file.read()

docs = json.loads(json_data)

In [4]:
# Note the number of documents in the dataset
len(docs)

20

In [5]:
# Preview a document to understand its structure
docs[0]

{'updated': '2024-05-20T17:30:49.148Z',
 'metadata': {'contentType': None,
  'productName': 'MongoDB Atlas',
  'tags': ['atlas', 'docs'],
  'version': None},
 'action': 'created',
 'sourceName': 'snooty-cloud-docs',
 'body': '# View Database Access History\n\n- This feature is not available for `M0` free clusters, `M2`, and `M5` clusters. To learn more, see Atlas M0 (Free Cluster), M2, and M5 Limits.\n\n- This feature is not supported on Serverless instances at this time. To learn more, see Serverless Instance Limitations.\n\n## Overview\n\nAtlas parses the MongoDB database logs to collect a list of authentication requests made against your clusters through the following methods:\n\n- `mongosh`\n\n- Compass\n\n- Drivers\n\nAuthentication requests made with API Keys through the Atlas Administration API are not logged.\n\nAtlas logs the following information for each authentication request within the last 7 days:\n\n<table>\n<tr>\n<th id="Field">\nField\n\n</th>\n<th id="Description">\nD

# Step 3: Chunk and embed the data


In [6]:
# You might see a warning after running this cell-- You can ignore it
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import Dict, List
import voyageai
from tqdm import tqdm

In [7]:
# Common list of separators for text data
separators = ["\n\n", "\n", " ", "", "#", "##", "###"]

In [8]:
# Use the `RecursiveCharacterTextSplitter` from LangChain to first split a piece of text on the list of `separators` above.
# Then recursively merge them into tokens until the specified chunk size is reached.
# For text data, you typically want to keep 1-2 paragraphs (~200 tokens) in a single chunk.
# Set chunk overlap to 0 for contextualized embeddings, otherwise 15-20% of chunk size.
# The `model_name` parameter indicates which encoder to use for tokenization, in this case GPT-4's encoder.
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name="gpt-4", separators=separators, chunk_size=200, chunk_overlap=0
)

📚 https://reference.langchain.com/python/langchain-text-splitters/character/RecursiveCharacterTextSplitter/split_text

In [9]:
# Define a function to split long documents into smaller chunks
def get_chunks(doc: Dict, text_field: str) -> List[Dict]:
    """
    Chunk up a document.

    Args:
        doc (Dict): Parent document to generate chunks from.
        text_field (str): Text field to chunk.

    Returns:
        List[Dict]: List of chunked documents.
    """
    # Extract the field to chunk from `doc`
    text = doc[text_field]
    # Split `text` using the `split_text` method of the `text_splitter` object above
    chunks = text_splitter.split_text(text)
    return chunks

In [10]:
# Initialize the Voyage AI client
voyageai.api_key = os.environ.get("VOYAGE_API_KEY")
vo = voyageai.Client()

📚 https://www.mongodb.com/docs/voyageai/models/contextualized-chunk-embeddings/?client=python#example

In [11]:
# Define a function to generate contextualized embeddings using the Voyage API
def get_embeddings(content: List[str], input_type: str) -> List[float] | List[List[float]]:
    """
    Get contextualized embeddings for each chunk.

    Args:
        content (List[str]): List of chunked texts or the user query as a list
        input_type (str): Type of input, either "document" or "query" 

    Returns:
        List[float] | List[List[float]]: Contextualized embeddings
    """
    # Use the `contextualized_embed` method of the Voyage API to get contextualized embeddings for each chunk with the following arguments:
    # inputs: `content` wrapped in another list
    # model: `voyage-context-4`
    # input_type: `input_type`
    embds_obj = vo.contextualized_embed(inputs=[content], model="voyage-context-4", input_type=input_type)
     # If `input_type` is "document", there is a single result with multiple embeddings, one for each chunk
    if input_type == "document":
        embeddings = [emb for r in embds_obj.results for emb in r.embeddings]
    # If `input_type` is "query", there is a single result with a single embedding
    if input_type == "query":
        embeddings = embds_obj.results[0].embeddings[0]
    return embeddings

In [12]:
embedded_docs = []
# Iterate through `docs` from Step 2
for doc in tqdm(docs):
    # Use the `get_chunks` function to chunk up the "body" field in each document
    chunks = get_chunks(doc, "body")
    # Pass all the `chunks` to the `get_embeddings` function to get contextualized embeddings for each chunk
    # `input_type` should be set to "document" since we are embedding the "documents" for RAG
    chunk_embeddings = get_embeddings(chunks, "document")
    # For each chunk, create a new document with the original metadata
    # Replace the `body` with the chunk content and add an `embedding` field
    for chunk, embedding in zip(chunks, chunk_embeddings):
        # Create a new document by copying the original document
        chunk_doc = doc.copy()
        # Replace the `body` field of `chunk_doc` with the chunk content
        chunk_doc["body"] = chunk
        # Add an `embedding` field to `chunk_doc`, containing the embedding for this chunk
        chunk_doc["embedding"] = embedding
        # Append `chunk_doc` to `embedded_docs`
        embedded_docs.append(chunk_doc)

100%|██████████| 20/20 [00:03<00:00,  5.82it/s]


In [13]:
# Notice that the length of `embedded_docs` is greater than the length of `docs` from Step 2 above
# This is because each document in `docs` has been split into multiple chunks
len(embedded_docs)

101

In [14]:
# Preview a chunked document to understand its structure
# Note that the structure looks similar to the original docs, except the `body` field now contains smaller chunks of text
# Each document also has an additional `embedding` field
embedded_docs[0]

{'updated': '2024-05-20T17:30:49.148Z',
 'metadata': {'contentType': None,
  'productName': 'MongoDB Atlas',
  'tags': ['atlas', 'docs'],
  'version': None},
 'action': 'created',
 'sourceName': 'snooty-cloud-docs',
 'body': '# View Database Access History\n\n- This feature is not available for `M0` free clusters, `M2`, and `M5` clusters. To learn more, see Atlas M0 (Free Cluster), M2, and M5 Limits.\n\n- This feature is not supported on Serverless instances at this time. To learn more, see Serverless Instance Limitations.\n\n## Overview\n\nAtlas parses the MongoDB database logs to collect a list of authentication requests made against your clusters through the following methods:\n\n- `mongosh`\n\n- Compass\n\n- Drivers\n\nAuthentication requests made with API Keys through the Atlas Administration API are not logged.\n\nAtlas logs the following information for each authentication request within the last 7 days:\n\n<table>\n<tr>\n<th id="Field">\nField\n\n</th>\n<th id="Description">\nD

# Step 4: Ingest data into MongoDB


### **Do not change the values assigned to the variables below**

In [15]:
# Name of the database -- Change if needed or leave as is
DB_NAME = "mongodb_genai_devday_rag"
# Name of the collection -- Change if needed or leave as is
COLLECTION_NAME = "knowledge_base"
# Name of the vector search index -- Change if needed or leave as is
ATLAS_VECTOR_SEARCH_INDEX_NAME = "vector_index"

In [16]:
# Connect to the `COLLECTION_NAME` collection.
collection = mongodb_client[DB_NAME][COLLECTION_NAME]

In [17]:
# Bulk delete all existing records from the collection defined above
collection.delete_many({})

DeleteResult({'n': 101, 'electionId': ObjectId('7fffffff000000000000020f'), 'opTime': {'ts': Timestamp(1788891798, 17), 't': 527}, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1788891798, 17), 'signature': {'hash': b'\xd1\x92K\\\x92(p\x03\x97S|\xe2\x9a\x1a\x8b~(\xa0\x9bY', 'keyId': 7647217351024181255}}, 'operationTime': Timestamp(1788891798, 17)}, acknowledged=True)

📚 https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.insert_many

In [18]:
# Bulk insert `embedded_docs` into the `collection` defined above -- should be a one-liner
collection.insert_many(embedded_docs)

print(f"Ingested {collection.count_documents({})} documents into the {COLLECTION_NAME} collection.")

Ingested 101 documents into the knowledge_base collection.


# Step 5: Create a vector search index

In [19]:
from utils import create_search_index, check_index_ready

In [20]:
# Create vector index definition specifying:
# path: Path to the embeddings field
# numDimensions: Number of embedding dimensions- depends on the embedding model used
# similarity: Similarity metric. One of cosine, euclidean, dotProduct.
model = {
    "name": ATLAS_VECTOR_SEARCH_INDEX_NAME,
    "type": "vectorSearch",
    "definition": {
        "fields": [
            {
                "type": "vector",
                "path": "embedding",
                "numDimensions": 1024,
                "similarity": "cosine",
            }
        ]
    },
}

In [21]:
# Use the `create_search_index` function from the `utils` module to create a vector search index with the above definition for the `collection` collection
create_search_index(collection, ATLAS_VECTOR_SEARCH_INDEX_NAME, model)

Creating vector_index index...
Successfully created vector_index index


In [22]:
# Use the `check_index_ready` function from the `utils` module to verify that the index was created and is in READY status before proceeding
check_index_ready(collection, ATLAS_VECTOR_SEARCH_INDEX_NAME)

vector_index index status: PENDING
vector_index index status: PENDING
vector_index index status: PENDING
vector_index index is READY


# Step 6: Perform vector search on your data


### Define a vector search function

📚 https://www.mongodb.com/docs/vector-search/query/aggregation-stages/vector-search-stage/?deployment-type=atlas&embedding=byo&interface=driver&language=python#simple-query-5


In [23]:
# Define a function to retrieve relevant documents for a user query using vector search
def vector_search(user_query: str) -> List[Dict]:
    """
    Retrieve relevant documents for a user query using vector search.

    Args:
    user_query (str): The user's query string.

    Returns:
    list: A list of matching documents.
    """

    # Generate embedding for the `user_query` using the `get_embeddings` function defined in Step 3
    # NOTE: Wrap the user_query in a list since the function expects a list of strings
    # `input_type` should be set to "query" since we are embedding the query
    query_embedding = get_embeddings([user_query], "query")

    # Define an aggregation pipeline consisting of a $vectorSearch stage, followed by a $project stage
    # Set the number of candidates to 150 and only return the top 5 documents from the vector search
    # In the $project stage, exclude the `_id` field and include these fields: `body`, `metadata.productName`, `metadata.contentType`, `updated` and `vectorSearchScore`
    # NOTE: Use variables defined previously for the `index`, `queryVector` and `path` fields in the $vectorSearch stage
    pipeline = [
        {
            "$vectorSearch": {
                "index": ATLAS_VECTOR_SEARCH_INDEX_NAME,
                "queryVector": query_embedding,
                "path": "embedding",
                "numCandidates": 150,
                "limit": 5
            }
        },
        {
            "$project": {
                "_id": 0,
                "body": 1,
                "metadata.productName": 1,
                "metadata.contentType": 1,
                "updated": 1,
                "score": {"$meta": "vectorSearchScore"}
            }
        }
    ]

    # Execute the aggregation `pipeline` and store the results in `results`
    results = collection.aggregate(pipeline)
    return list(results)

### Run vector search queries


In [24]:
vector_search("What are some best practices for data backups in MongoDB?")

[{'updated': '2024-05-20T17:31:07.735Z',
  'metadata': {'contentType': None, 'productName': 'MongoDB Server'},
  'body': '# Backup and Restore Sharded Clusters\n\nThe following tutorials describe backup and restoration for sharded clusters:\n\nTo use `mongodump` and `mongorestore` as a backup strategy for sharded clusters, you must stop the sharded cluster balancer and use the `fsync` command or the `db.fsyncLock()` method on `mongos` to block writes on the cluster during backups.\n\nSharded clusters can also use one of the following coordinated backup and restore processes, which maintain the atomicity guarantees of transactions across shards:\n\n- MongoDB Atlas\n\n- MongoDB Cloud Manager\n\n- MongoDB Ops Manager\n\nUse file system snapshots back up each component in the sharded cluster individually. The procedure involves stopping the cluster balancer. If your system configuration allows file system backups, this might be more efficient than using MongoDB tools.\n\nCreate backups usi

In [25]:
vector_search("How to resolve alerts in MongoDB?")

[{'updated': '2024-05-20T17:30:49.148Z',
  'metadata': {'contentType': None, 'productName': 'MongoDB Atlas'},
  'body': '</td>\n<td headers="Description">\nPercentage of used disk space on a partition reaches a specified threshold.\n\n</td>\n</tr>\n<tr>\n<td headers="Alert%20Type">\nQuery Targeting Alerts\n\n</td>\n<td headers="Description">\nIndicates inefficient queries.\n\nThe change streams cursors that the MongoDB Search process (`mongot`) uses to keep MongoDB Search indexes updated can contribute to the query targeting ratio and trigger query targeting alerts if the ratio is high.\n\n</td>\n</tr>\n<tr>\n<td headers="Alert%20Type">\nReplica Set Has No Primary\n\n</td>\n<td headers="Description">\nNo primary is detected in replica set.\n\n</td>\n</tr>\n<tr>\n<td headers="Alert%20Type">\nReplication Oplog Alerts\n\n</td>\n<td headers="Description">\nAmount of oplog data generated on a primary cluster member is larger than the cluster\'s configured oplog size.',
  'score': 0.79963815

# 🦹‍♀️ Combine pre-filtering with vector search

### Filter for documents where the product name is `MongoDB Atlas`

📚 https://www.mongodb.com/docs/vector-search/index/vector-search-type/#about-the-filter-type

In [26]:
# Modify the vector search index `model` from Step 6 to include the `metadata.productName` field as a `filter` field
model = {
    "name": ATLAS_VECTOR_SEARCH_INDEX_NAME,
    "type": "vectorSearch",
    "definition": {
        "fields": [
            {
                "type": "vector",
                "path": "embedding",
                "numDimensions": 1024,
                "similarity": "cosine"
            },
            {"type": "filter", "path": "metadata.productName"}
        ]
    }
}

In [27]:
# Use the `create_search_index` function from the `utils` module to re-create the vector search index with the modified model
create_search_index(collection, ATLAS_VECTOR_SEARCH_INDEX_NAME, model)

vector_index index exists, dropping...
vector_index index dropped
Creating vector_index index...
Successfully created vector_index index


In [28]:
# Use the `check_index_ready` function from the `utils` module to verify that the index has the right filter fields and is in READY status before proceeding
check_index_ready(collection, ATLAS_VECTOR_SEARCH_INDEX_NAME)

vector_index index status: PENDING
vector_index index status: PENDING
vector_index index status: PENDING
vector_index index is READY


In [29]:
# Embed the user query
query_embedding = get_embeddings(
    ["What are some best practices for data backups in MongoDB?"], "query"
)

📚 https://www.mongodb.com/docs/vector-search/query/aggregation-stages/vector-search-stage/?deployment-type=atlas&embedding=byo&interface=driver&language=python#filter-query-5

In [30]:
# Modify the aggregation pipeline defined in Step 6 to:
# Include a filter in the $vectorSearch stage for documents where the `metadata.productName` field has the value "MongoDB Atlas".
pipeline = [
    {
        "$vectorSearch": {
            "index": ATLAS_VECTOR_SEARCH_INDEX_NAME,
            "path": "embedding",
            "queryVector": query_embedding,
            "numCandidates": 150,
            "limit": 5,
            "filter": {"metadata.productName": "MongoDB Atlas"}
        }
    },
    {
        "$project": {
            "_id": 0,
            "body": 1,
            "metadata.productName": 1,
            "metadata.contentType": 1,
            "updated": 1,
            "score": {"$meta": "vectorSearchScore"}
        }
    }
]

In [31]:
# Execute the aggregation pipeline and view the results
results = collection.aggregate(pipeline)
list(results)

[{'updated': '2024-05-20T17:30:49.148Z',
  'metadata': {'contentType': None, 'productName': 'MongoDB Atlas'},
  'body': '</td>\n<td headers="Description">\nPercentage of used disk space on a partition reaches a specified threshold.\n\n</td>\n</tr>\n<tr>\n<td headers="Alert%20Type">\nQuery Targeting Alerts\n\n</td>\n<td headers="Description">\nIndicates inefficient queries.\n\nThe change streams cursors that the MongoDB Search process (`mongot`) uses to keep MongoDB Search indexes updated can contribute to the query targeting ratio and trigger query targeting alerts if the ratio is high.\n\n</td>\n</tr>\n<tr>\n<td headers="Alert%20Type">\nReplica Set Has No Primary\n\n</td>\n<td headers="Description">\nNo primary is detected in replica set.\n\n</td>\n</tr>\n<tr>\n<td headers="Alert%20Type">\nReplication Oplog Alerts\n\n</td>\n<td headers="Description">\nAmount of oplog data generated on a primary cluster member is larger than the cluster\'s configured oplog size.',
  'score': 0.61727148

### Filter on documents which have been updated on or after `2024-05-19` and where the content type is `Tutorial`

📚 https://www.mongodb.com/docs/vector-search/index/vector-search-type/#about-the-filter-type

In [32]:
# Modify the vector search index `model` from Step 6 to include `metadata.contentType` and `updated` as `filter` fields
model = {
    "name": ATLAS_VECTOR_SEARCH_INDEX_NAME,
    "type": "vectorSearch",
    "definition": {
        "fields": [
            {
                "type": "vector",
                "path": "embedding",
                "numDimensions": 1024,
                "similarity": "cosine"
            },
            {"type": "filter", "path": "metadata.contentType"},
            {"type": "filter", "path": "updated"}
        ]
    }
}

In [33]:
# Use the `create_search_index` function from the `utils` module to re-create the vector search index with the modified model
create_search_index(collection, ATLAS_VECTOR_SEARCH_INDEX_NAME, model)

vector_index index exists, dropping...
vector_index index dropped
Creating vector_index index...
Successfully created vector_index index


In [34]:
# Use the `check_index_ready` function from the `utils` module to verify that the index has the right filter fields and is in READY status before proceeding
check_index_ready(collection, ATLAS_VECTOR_SEARCH_INDEX_NAME)

vector_index index status: PENDING
vector_index index status: PENDING
vector_index index is READY


In [35]:
# Embed the user query
query_embedding = get_embeddings(
    ["What are some best practices for data backups in MongoDB?"], "query"
)

📚 https://www.mongodb.com/docs/vector-search/query/aggregation-stages/vector-search-stage/?deployment-type=atlas&embedding=byo&interface=driver&language=python#filter-query-5

In [36]:
# Modify the aggregation pipeline defined in Step 6 to:
# Include a filter in the $vectorSearch stage for documents where the `metadata.contentType` field is "Tutorial" AND the `updated` field is greater than or equal to "2024-05-19".
# HINT: Use the $gte operator to check >= and the $and operator to combine the conditions.
pipeline = [
    {
        "$vectorSearch": {
            "index": ATLAS_VECTOR_SEARCH_INDEX_NAME,
            "path": "embedding",
            "queryVector": query_embedding,
            "numCandidates": 150,
            "limit": 5,
            "filter": {
                "$and": [
                    {"metadata.contentType": "Tutorial"},
                    {"updated": {"$gte": "2024-05-19"}}
                ]
            }
        }
    },
    {
        "$project": {
            "_id": 0,
            "body": 1,
            "metadata.productName": 1,
            "metadata.contentType": 1,
            "updated": 1,
            "score": {"$meta": "vectorSearchScore"}
        }
    }
]

In [37]:
# Execute the aggregation pipeline and view the results
results = collection.aggregate(pipeline)
list(results)

[{'updated': '2024-05-20T17:32:23.500Z',
  'metadata': {'contentType': 'Tutorial', 'productName': None},
  'body': '- Give CodeWhisperer something to work with. The more code your file contains, the more context CodeWhisperer has for generating recommendations.\n - Write descriptive comments in natural language — for example\n```\n// Take a JSON document as a String and store it in MongoDB returning the _id\n```\nOr\n```\n//Insert a document in a collection with a given _id and a discountLevel\n```\n - Specify the libraries you prefer at the start of your file by using import statements.\n```\n// This Java class works with MongoDB sync driver.\n// This class implements Connection to MongoDB and CRUD methods.\n```\n - Use descriptive names for variables and functions\n - Break down complex tasks into simpler tasks\n\n**Provide feedback**\n----------------',
  'score': 0.6129699945449829},
 {'updated': '2024-05-20T17:32:23.500Z',
  'metadata': {'contentType': 'Tutorial', 'productName': N

# Step 7: Build the RAG application


In [38]:
from strands import Agent
from IPython.display import display, Markdown
from strands_xai import xAIModel

### Define a function to create the chat prompt

In [39]:
# Define the system prompt for the RAG application
SYSTEM_PROMPT = "You are a MongoDB documentation assistant. Answer the user's question using only the retrieved context provided with it. If the context does not contain the answer, say I DON'T KNOW."

In [47]:
# Define a function to create the user prompt for our RAG application
def create_prompt(user_query: str) -> str:
    """
    Create a chat prompt that includes the user query and retrieved context.

    Args:
        user_query (str): The user's query string.

    Returns:
        str: The chat prompt string.
    """
    # Retrieve the most relevant documents for the `user_query` using the `vector_search` function defined in Step 6
    context = vector_search(user_query)
    # Join the retrieved documents into a single string, where each document is separated by two new lines ("\n\n")
    context = "\n\n".join([doc.get('body') for doc in context])
    # Prompt consisting of the user query and relevant context to answer it
    prompt = f"Context:\n{context}\n\nQuestion:{user_query}"
    return prompt

### Define a function to answer user queries

In [57]:
# Initialize the Strands Agent with the system prompt
xai_key = os.environ.get("XAI_API_KEY")
model = xAIModel(
    client_args={"api_key": xai_key},
    model_id="grok-build-0.1",
)

agent = Agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
)

# Define a function to answer user queries using Strands Agent
def generate_answer(user_query: str) -> None:
    """
    Generate an answer to the user query.

    Args:
        user_query (str): The user's query string.
    """
    # Use the `create_prompt` function above to create a chat prompt with retrieved context
    prompt = create_prompt(user_query)
    # Send the prompt to the Strands Agent and get the response
    response = agent(prompt)
    display(Markdown(str(response)))

### Query the RAG application


In [58]:
generate_answer("What are some best practices for data backups in MongoDB?")

The question is: "What are some best practices for data backups in MongoDB?"
Best practices for data backups in MongoDB, based on the provided documentation, focus on sharded clusters and coordinated tools. For sharded setups, stop the balancer and use `fsync` or `db.fsyncLock()` on `mongos` to block writes during `mongodump`/`mongorestore` backups. Consider MongoDB Atlas, Cloud Manager, or Ops Manager for atomicity across shards. File system snapshots per component (after stopping the balancer) or individual `mongodump` operations can also work, with balancer limits creating backup windows. For full cluster restores, outline procedures accordingly.**Best practices for data backups in MongoDB (from the provided context, focused on sharded clusters):**

- **For `mongodump` and `mongorestore`**: Stop the sharded cluster balancer and use the `fsync` command or `db.fsyncLock()` method on `mongos` to block writes on the cluster during backups. This ensures a consistent backup state.
- **Coo

**Best practices for data backups in MongoDB (from the provided context, focused on sharded clusters):**

- **For `mongodump` and `mongorestore`**: Stop the sharded cluster balancer and use the `fsync` command or `db.fsyncLock()` method on `mongos` to block writes on the cluster during backups. This ensures a consistent backup state.
- **Coordinated backup/restore for sharded clusters**: Use MongoDB Atlas, MongoDB Cloud Manager, or MongoDB Ops Manager. These maintain the atomicity guarantees of transactions across shards (unlike basic per-component tools).
- **File system snapshots**: Back up each component in the sharded cluster individually. The procedure requires stopping the cluster balancer. This can be more efficient than MongoDB tools if your system configuration supports it.
- **Using `mongodump`**: Create backups by running `mongodump` to back up each component in the cluster individually.
- **Balancer management**: Limit the operation of the cluster balancer to create a window for regular backup operations.
- **Restoration considerations**: Follow the outlined procedures and considerations for restoring an *entire* sharded cluster from backup.

The provided context does not cover general (non-sharded) clusters, other backup methods (e.g., replication-based), or broader best practices beyond sharded cluster scenarios.


# 🦹‍♀️ Re-rank retrieved results


📚 https://www.mongodb.com/docs/voyageai/models/rerankers/?client=python#example

In [59]:
# Add a re-ranking step to the following function
def create_prompt(user_query: str) -> str:
    """
    Create a chat prompt that includes the user query and retrieved context.

    Args:
        user_query (str): The user's query string.

    Returns:
        str: The chat prompt string.
    """
    # Retrieve the most relevant documents for the `user_query` using the `vector_search` function defined in Step 6
    context = vector_search(user_query)
    # Extract the "body" field from each document in `context`
    documents = [d.get("body") for d in context]
    # Use the `rerank` method of the Voyage API to re-rank the `documents` with the following arguments:
    # model: "rerank-2.5"
    # top_k: 5
    reranked_documents = vo.rerank(user_query, documents, model="rerank-2.5", top_k=5)
    # Join the re-ranked documents into a single string, where each document is separated by two new lines ("\n\n")
    context = "\n\n".join([d.document for d in reranked_documents.results])
    # Prompt consisting of the user query and relevant context to answer it
    prompt = f"Context:\n{context}\n\nQuestion:{user_query}"
    return prompt

In [60]:
# Note the impact of re-ranking on the generated answer
# You might not see a difference in this example since we are only re-ranking 5 documents
# In practice, you would send a larger number of documents to the re-ranker, and get the top few AFTER reranking
generate_answer("What are some best practices for data backups in MongoDB?")

The user's question is: "What are some best practices for data backups in MongoDB?"
Some best practices for data backups in MongoDB, based on the provided context, focus on sharded clusters and include:

- Using `mongodump` and `mongorestore` as a backup strategy, but stopping the sharded cluster balancer and using the `fsync` command or `db.fsyncLock()` on `mongos` to block writes during backups.
- Leveraging coordinated backup and restore processes like MongoDB Atlas, MongoDB Cloud Manager, or MongoDB Ops Manager to maintain atomicity guarantees across shards.
- Employing file system snapshots to back up each component individually, which can be more efficient if your system allows it (after stopping the cluster balancer).
- Creating backups using `mongodump` for each component in the cluster individually.
- Limiting the operation of the cluster balancer to create a window for regular backup operations.
- For restoring an entire sharded cluster, follow an outline that considers the b

**Best practices for data backups in MongoDB (drawn directly from the provided context on sharded clusters):**

- To use `mongodump` and `mongorestore` as a backup strategy for sharded clusters, you must stop the sharded cluster balancer and use the `fsync` command or the `db.fsyncLock()` method on `mongos` to block writes on the cluster during backups.
- Sharded clusters can use coordinated backup and restore processes that maintain atomicity guarantees of transactions across shards: MongoDB Atlas, MongoDB Cloud Manager, or MongoDB Ops Manager.
- Use file system snapshots to back up each component in the sharded cluster individually. The procedure involves stopping the cluster balancer. If your system configuration allows file system backups, this might be more efficient than using MongoDB tools.
- Create backups using `mongodump` to back up each component in the cluster individually.
- Limit the operation of the cluster balancer to provide a window for regular backup operations.
- Follow the outlined procedure and considerations for restoring an *entire* sharded cluster from backup.

The context focuses exclusively on sharded clusters and does not provide general (non-sharded) backup practices.
